# 28 AG RUN-3: the runs without the REC export double count

**Purpose.** To find out how much of the RPS gap in the stage-1 ledger comes from the double count of REC exports
in ReEDS (observation O27; `docs/rps_residual_note.md`; review page Decisions, item 3), this notebook compares the
RUN-3 runs with the runs they copy. RUN-3 sets `GSw_RECExportFix = 1`, which removes the three terms of
`eq_REC_Requirement` that subtract exports a second time from the seller's RECs (`reeds/core/setup/c_model.gms`;
master plan G5 and D8). Ethan agreed the run set on 2026-11-04 (Decisions, item 12): 7 fix runs (`recfix_*`, the
no-mandate baseline `fbR_none` and the six P50 SMR mandate runs) and 2 controls (`recctl_fbR_none`,
`recctl_smr100_eo_p50`) that run today's code with the switch off.

The notebook reads only extracts (notebook 20 §6 for the sources, §9 for batch `ag`). It has four parts:

1. **Reproduction.** Each control against its archived source run. A control that equals its source shows that a
   fix run minus its source is the fix alone, not a change in the code between the two checkouts.
2. **Fix against source, per run.** The REC trade wedge by origin, REC quantities, RPS payments, system cost,
   emissions, the build mix and the SMR support (paper 1's shadow price), and the ledger of each fix run against its
   source.
3. **The stage-1 ledger with and without the fix.** Each mandate pair (SMR run minus `fbR_none`) built from the
   source runs and from the fix runs, with the same code (`aglib.ledger.pair`) as notebook 21.
4. **Exports** for the review page.

Money is 2025 $ unless a cell says otherwise; PVs are to 2026. The notebook logs no interpretation (protocol §10).

In [1]:
import sys
from pathlib import Path
_p = Path.cwd().resolve()
while not (_p / "nlpaper").is_dir():           # find z-ethan/ from wherever the kernel starts
    _p = _p.parent
sys.path.insert(0, str(_p))
import numpy as np
import pandas as pd

import nlpaper
from nlpaper import paths, plotstyle as ps
from aglib import ledger as L
nlpaper.setup(threads=4)
pd.set_option("display.width", 220)
pd.set_option("display.max_rows", 300)
pd.set_option("display.max_columns", 30)
OUT = paths.exports("28")
TO25 = ps.to2025(2004)
print(f"exports to {paths.rel(OUT)}; 2004 $ -> 2025 $ x {TO25:.4f}")

nlpaper setup: python 3.12.3 | numpy 1.26.4 | pandas 2.2.2 | matplotlib 3.8.4 | threads 4
determinism: seeds are explicit numpy Generators in each cell; results do not depend on PYTHONHASHSEED.
exports to z-ethan\results\exports\28; 2004 $ -> 2025 $ x 1.7021


## 1. The runs

`SRC` holds the archived source run of each RUN-3 case, `FIX` the fix run and `CTRL` the two controls.
`Y` is the stage-1 window, 2026–2050.

In [2]:
TARGETS = ["eia", "aj", "iaea", "mck", "cop28", "eo"]
NAMES = ["fbR_none"] + TARGETS
SRC = {"fbR_none": ("itcfb", "fbR_none"), **{t: ("smr100", f"smr100_{t}_p50") for t in TARGETS}}
FIX = {"fbR_none": ("ag", "recfix_fbR_none"), **{t: ("ag", f"recfix_smr100_{t}_p50") for t in TARGETS}}
CTRL = {"fbR_none": ("ag", "recctl_fbR_none"), "eo": ("ag", "recctl_smr100_eo_p50")}
Y = (2026, 2050)
EXT = L.load_external()
CRF = L.crf_after_2026()
rd = lambda b, k: pd.read_parquet(L.EXTRACTS_AG / b / f"{k}.parquet")
ALL = {**{f"source {k}": v for k, v in SRC.items()}, **{f"fix {k}": v for k, v in FIX.items()},
       **{f"control {k}": v for k, v in CTRL.items()}}
for name, (b, c) in ALL.items():
    assert c in set(rd(b, "derived_ledger")["case"]), f"{b}/{c} has no derived ledger"
    assert c in set(rd(b, "derived_recs")["case"]), f"{b}/{c} has no REC split"
print(f"all {len(ALL)} runs have derived extracts")

all 16 runs have derived extracts


### The summary quantities of one run

`summary` returns one row of quantities per run, each summed over 2026–2050 in the units given: every item of
the derived ledger (2004 $B, or TWh for `Q_*` items), total system cost (all `systemcost` rows, and without the
credit rows; 2004 $B), process CO2 (Mt), the REC trade wedge by origin (2004 $B) and the REC quantities by origin
(TWh), and the capacity of each owner group in 2050 (GW). The SMR support in each solve year is the mandate's shadow
price × `pvf_onm` (2004 $ per MW, as in notebook 21's check C4); `support_mean` is its mean over the years where it
is positive.

In [3]:
def _win(d):
    return d[(d["t"].astype(int) >= Y[0]) & (d["t"].astype(int) <= Y[1])]

def summary(batch, case):
    s = {}
    der = _win(L._sel(batch, "derived_ledger", case))
    for item, v in der.groupby("item")["Value"].sum().items():
        s[f"ledger|{item}"] = v / (1e6 if item.startswith("Q_") else 1e9)
    sc = _win(L._sel(batch, "systemcost", case))
    s["system cost (all rows)"] = sc["Value"].sum() / 1e9
    s["system cost (no credit rows)"] = sc.loc[~sc["sys_costs"].str.contains(L.CREDIT), "Value"].sum() / 1e9
    em = _win(L._sel(batch, "emit_nat", case))
    s["CO2 process, Mt"] = em.loc[(em["eall"] == "CO2") & (em["etype"] == "process"), "Value"].sum() / 1e6
    rc = _win(L._sel(batch, "derived_recs", case))
    for item, v in rc.groupby("item")["Value"].sum().items():
        if item.startswith("wedge|"):
            s[f"recs|{item}"] = v / 1e9
        elif item.startswith("Q|"):
            s[f"recs|{item}"] = v / 1e6
    s["recs|wedge|total"] = sum(v for k, v in s.items() if k.startswith("recs|wedge|"))
    cap = L._sel(batch, "cap_nat", case)
    cap = cap[cap["t"].astype(int) == 2050]
    for g, v in cap.groupby(cap["i"].map(L.group))["Value"].sum().items():
        s[f"cap 2050 GW|{g}"] = v / 1e3
    _, memo = L.accounts_cached(batch, case, EXT)
    sup = memo["dual"] * memo["pvf"]
    s["support_mean, 2004 $/MW"] = float(sup[sup > 0].mean()) if (sup > 0).any() else 0.0
    return pd.Series(s)

SUM = {}
for k, r in SRC.items():
    SUM[("source", k)] = summary(*r)
for k, r in FIX.items():
    SUM[("fix", k)] = summary(*r)
for k, r in CTRL.items():
    SUM[("control", k)] = summary(*r)
SUM = pd.DataFrame(SUM).fillna(0.0)
print(SUM.shape)

(33, 16)


## 2. Reproduction: each control against its source

The table lists every summary quantity whose control and source differ by more than 0.01% of the source (or of
1 unit where the source is near zero), with the fix run's difference beside it for scale. The next cell builds the
ledger of each control against its source (`L.pair` with no mandate support), which shows how far the ledger moves
when nothing but the checkout changes.

In [4]:
rows = []
for k in CTRL:
    src, ctl, fix = SUM[("source", k)], SUM[("control", k)], SUM[("fix", k)]
    den = src.abs().clip(lower=1.0)
    rows.append(pd.DataFrame({"run": k, "source": src, "control": ctl, "fix": fix,
                              "control - source, %": 100 * (ctl - src) / den,
                              "fix - source, %": 100 * (fix - src) / den}))
REPRO = pd.concat(rows).rename_axis("quantity").reset_index()
print("largest |control - source| over all quantities, % of the source:",
      REPRO.groupby("run")["control - source, %"].apply(lambda s: s.abs().max()).round(4).to_dict())
big = REPRO[REPRO["control - source, %"].abs() > 0.01]
print(f"{len(big)} of {len(REPRO)} quantities differ by more than 0.01%:")
print(big.round(4).to_string(index=False))
print("\nthe REC wedge by origin:")
print(REPRO[REPRO["quantity"].str.startswith("recs|wedge")].round(4).to_string(index=False))

largest |control - source| over all quantities, % of the source: {'eo': 4.0019, 'fbR_none': 1.9637}
5 of 66 quantities differ by more than 0.01%:
               quantity      run    source   control       fix  control - source, %  fix - source, %
        ledger|CR_C_vsc fbR_none    0.6542    0.6537    0.5304              -0.0582         -12.3870
  recs|Q|seller_unbound fbR_none 4286.9208 4202.7373 4494.5242              -1.9637           4.8427
        ledger|CR_C_vsc       eo    0.3857    0.3855    0.3762              -0.0265          -0.9576
  recs|Q|seller_unbound       eo 4305.2395 4477.5289 5367.8786               4.0019          24.6825
recs|wedge|double_count       eo    1.6451    1.6449    0.4531              -0.0140         -72.4583

the REC wedge by origin:
                 quantity      run  source  control     fix  control - source, %  fix - source, %
  recs|wedge|double_count fbR_none  1.4585   1.4585  0.5398               0.0050         -62.9891
recs|wedge|seller_unbound 

In [5]:
FREE = lambda d: d[d["allowance_state"].isin(["free", "none"]) & d["allowance_national"].isin(["free", "none"])]

def actor_sums(out, mandate, col="PV_2.0%", horizon="A"):
    ap = FREE(L.actor_pvs(out, mandate))
    if mandate:
        ap = ap[(ap["funding"] == "T") & (ap["support_form"] == "lump")]
    ap = ap[(ap["horizon"] == horizon) & (ap["actor"] != "environment")]   # one entry per carbon basis and health model: never summed
    return ap.groupby("actor")[col].sum()          # L.actor_pvs reports 2025 $B

CTRL_LEDGER = {k: actor_sums(L.pair(CTRL[k], SRC[k], EXT, CRF, mandate=False), False) for k in CTRL}
CL = pd.DataFrame(CTRL_LEDGER)
print("ledger of control minus source, horizon A, PV at 2.0%, 2025 $B; largest |actor sum| per run:",
      CL.abs().max().round(4).to_dict())
print("for scale, largest |actor sum| of the fix against its source:", {k: round(float(actor_sums(L.pair(FIX[k], SRC[k], EXT, CRF, mandate=False), False).abs().max()), 3) for k in CTRL})
print(CL.round(4).to_string())

ledger of control minus source, horizon A, PV at 2.0%, 2025 $B; largest |actor sum| per run: {'fbR_none': 0.0454, 'eo': 0.4829}


for scale, largest |actor sum| of the fix against its source: {'fbR_none': 15.994, 'eo': 19.676}
                                   fbR_none      eo
actor                                              
electricity consumers                0.0454  0.0174
foreign buyers                      -0.0007 -0.0003
foreign sellers: Canadian imports   -0.0003 -0.2043
gas producers                        0.0015  0.0006
network                              0.0010 -0.0075
non-power gas consumers             -0.0009 -0.0003
owners: H2 and DAC production       -0.0000  0.0000
owners: SMR (new)                    0.0000 -0.0045
owners: coal                        -0.0161 -0.3218
owners: existing nuclear            -0.0134  0.0061
owners: gas                         -0.0148 -0.0089
owners: hydro                        0.0020  0.0242
owners: other                        0.0011  0.0007
owners: storage                      0.0000 -0.0021
owners: wind + solar                -0.0026  0.4829
taxpayers          

## 3. Each fix run against its source

**The REC trade wedge by origin**, 2026–2050, 2004 $B, and the REC quantities (TWh). The split is the rule of
`aglib.reduce.reduce_run_recs`: on RECs from a seller with a positive price, `double_count` is
min(p_dst − p_src, p_src) per REC by construction. In the fix runs the double count is gone from the model, so the
part carrying that label there is a price gap that the rule assigns, not the model error.

In [6]:
W = SUM.loc[[i for i in SUM.index if i.startswith("recs|")]].T
W.columns = [c.replace("recs|", "") for c in W.columns]
W = W[[c for c in ["wedge|seller_unbound", "wedge|double_count", "wedge|trade_limits", "wedge|voluntary", "wedge|total",
                   "Q|seller_unbound", "Q|seller_priced", "Q|voluntary"] if c in W.columns]]
print(W.round(3).to_string())
d = (W.xs("fix", level=0) - W.xs("source", level=0)).loc[NAMES]
print("\nfix - source:")
print(d.round(3).to_string())
print("\nfix - source, % of the source total wedge:")
print((100 * d.filter(like="wedge|").div(W.xs("source", level=0).loc[NAMES, "wedge|total"], axis=0)).round(1).to_string())

                  wedge|seller_unbound  wedge|double_count  wedge|trade_limits  wedge|total  Q|seller_unbound  Q|seller_priced
source  fbR_none                10.742               1.458               0.502       12.703          4286.921          447.401
        eia                     10.807               1.455               0.487       12.749          4491.753          455.300
        aj                      10.370               1.465               0.481       12.316          4169.427          439.097
        iaea                    10.396               1.357               0.283       12.036          4519.251          416.961
        mck                     10.572               1.371               0.284       12.227          4418.886          435.617
        cop28                   10.796               1.425               0.257       12.478          4523.027          469.810
        eo                      11.519               1.645               0.212       13.376          4305.239  

In [7]:
mr = pd.concat([rd(b, "derived_recs").query("case == @c").assign(run=k, who=w)
                for w, D_ in (("source", SRC), ("fix", FIX)) for k, (b, c) in D_.items()])
mr = mr[(mr["item"] == "min_ratio|seller_priced") & mr["t"].between(*Y)]
print("smallest buyer/seller price ratio on RECs from priced sellers, 2026-2050:")
print(mr.groupby(["run", "who"])["Value"].min().unstack("who")[["source", "fix"]].loc[NAMES].round(3).to_string())

smallest buyer/seller price ratio on RECs from priced sellers, 2026-2050:
who       source    fix
run                    
fbR_none   2.000  1.000
eia        2.000  1.000
aj         2.000  1.000
iaea       1.995  1.000
mck        1.995  1.000
cop28      1.993  1.000
eo         1.993  0.944


**The other quantities.** RPS payments (`P_R`), RECs bought and received (`REC_paid`, `REC_recv`), energy and
capacity payments, system cost, process CO2 and the SMR support, for each run, source and fix.

In [8]:
keys = ["ledger|P_R", "ledger|REC_paid", "ledger|REC_recv", "ledger|P_E", "ledger|P_C",
        "system cost (no credit rows)", "system cost (all rows)", "CO2 process, Mt", "support_mean, 2004 $/MW"]
keys = [k for k in keys if k in SUM.index]
T = []
for k in NAMES:
    s, f = SUM[("source", k)][keys], SUM[("fix", k)][keys]
    T.append(pd.DataFrame({"run": k, "source": s, "fix": f, "fix - source": f - s,
                           "%": 100 * (f - s) / s.abs().clip(lower=1e-9)}))
OTHER = pd.concat(T).rename_axis("quantity").reset_index()
print(OTHER.round(3).to_string(index=False))

                    quantity      run      source         fix  fix - source       %
                  ledger|P_R fbR_none      70.596      58.457       -12.139 -17.195
             ledger|REC_paid fbR_none      72.038      58.419       -13.619 -18.906
             ledger|REC_recv fbR_none      59.335      47.485       -11.850 -19.971
                  ledger|P_E fbR_none    1519.268    1533.375        14.107   0.929
                  ledger|P_C fbR_none    1523.107    1515.053        -8.054  -0.529
system cost (no credit rows) fbR_none    3508.197    3490.437       -17.761  -0.506
      system cost (all rows) fbR_none    3208.452    3195.031       -13.422  -0.418
             CO2 process, Mt fbR_none    9993.574   10102.535       108.961   1.090
     support_mean, 2004 $/MW fbR_none       0.000       0.000         0.000   0.000
                  ledger|P_R      eia      66.564      52.910       -13.654 -20.512
             ledger|REC_paid      eia      67.955      52.820       -15.136 

In [9]:
capk = [i for i in SUM.index if i.startswith("cap 2050 GW|")]
CAP = (SUM.xs("fix", axis=1, level=0)[NAMES] - SUM.xs("source", axis=1, level=0)[NAMES]).loc[capk]
CAP.index = [i.split("|", 1)[1] for i in CAP.index]
print("capacity in 2050, fix - source, GW:")
print(CAP[(CAP.abs() > 0.05).any(axis=1)].round(2).to_string())

capacity in 2050, fix - source, GW:
                       fbR_none   eia    aj  iaea   mck  cop28    eo
H2 and DAC production     -0.06 -0.06 -0.01  0.01  0.06   0.06  0.11
coal                      -0.94 -0.55  0.11  0.35  0.31   0.66 -1.27
gas                        5.21  3.00 -0.62  1.20 -0.76   2.72  5.69
hydro                      0.06  0.06  0.01  0.02  0.02  -0.02 -0.02
other                     -0.04 -0.09 -0.01  0.00  0.16   0.32  0.46
storage                   -5.80 -4.03  0.76 -0.10  0.25   0.75 -2.07
wind + solar              -6.76 -4.29  2.52 -3.50 -3.25 -13.16 -9.15


**The ledger of each fix run against its source** (`L.pair` with no mandate support; the SMR support is compared
above). Horizon A, PV at 2.0%, actor sums, allowances free, 2025 $B.

In [10]:
FIXLED = {k: actor_sums(L.pair(FIX[k], SRC[k], EXT, CRF, mandate=False), False) for k in NAMES}
FL = pd.DataFrame(FIXLED)[NAMES]
FL = FL[(FL.abs() > 0.005).any(axis=1)]
print(FL.round(2).to_string())

                                   fbR_none    eia     aj   iaea   mck  cop28     eo
actor                                                                               
electricity consumers                 15.99  12.11  -5.54  -3.58 -4.62 -21.88 -19.68
foreign buyers                        -3.55  -6.43  -0.44  -0.60 -2.80  -4.69  -0.40
foreign sellers: Canadian imports     -0.59  -0.74  -0.39  -1.00 -1.21  -0.64  -0.59
gas producers                         11.35  20.68   0.62   1.44  8.99  15.11   0.89
network                               -4.09  -5.32  -2.33   0.43  0.39   6.44   3.75
non-power gas consumers               -4.87  -8.86  -0.25  -0.47 -3.61  -6.45  -0.25
owners: H2 and DAC production          0.11   0.10  -0.09  -0.09 -0.32  -0.30  -0.44
owners: SMR (new)                      0.00   0.22  -0.09   0.56  0.84   2.23   2.54
owners: coal                          -1.90  -2.17  -0.48  -0.80 -1.42  -0.04  -0.56
owners: existing nuclear               2.41   2.62   1.50   0.39 

## 4. The stage-1 ledger with and without the fix

Each mandate pair is built twice with notebook 21's code: from the source runs (`smr100_{t}_p50` minus
`fbR_none`, as in notebook 21) and from the fix runs (`recfix_smr100_{t}_p50` minus `recfix_fbR_none`). The tables
show funding case T, the lump support form, allowances free, horizon A, PV at 2.0%. The first table sums each
actor's entries (the environment is left out: it holds one entry per carbon basis and health model, which never sum;
the views below carry it); the second lists the RPS-related entries; the third shows the two resident views in the reference
variant (AP2 ACS, MCPF 1, φ = 0) at the global EPA 2023 carbon value at 2.0%.

In [11]:
PAIR_SRC = {t: L.pair(SRC[t], SRC["fbR_none"], EXT, CRF) for t in TARGETS}
PAIR_FIX = {t: L.pair(FIX[t], FIX["fbR_none"], EXT, CRF) for t in TARGETS}
A_src = pd.DataFrame({t: actor_sums(o, True) for t, o in PAIR_SRC.items()})[TARGETS]
A_fix = pd.DataFrame({t: actor_sums(o, True) for t, o in PAIR_FIX.items()})[TARGETS]
A_d = A_fix - A_src
keep = (A_src.abs() > 0.05).any(axis=1) | (A_d.abs() > 0.05).any(axis=1)
for name, tab in (("source pairs", A_src), ("fix pairs", A_fix), ("fix - source", A_d)):
    print(f"\n{name}, actor sums, 2025 $B:")
    print(tab[keep].round(2).to_string())


source pairs, actor sums, 2025 $B:
                                     eia     aj    iaea     mck   cop28      eo
actor                                                                          
electricity consumers              32.70  51.14  133.36  162.56  353.37  585.90
foreign buyers                      7.84   7.66   21.62   28.93   62.04   89.89
foreign sellers: Canadian imports  -1.20  -1.38   -3.05   -2.98   -4.88   -6.17
gas producers                     -26.88 -26.16  -74.59  -98.96 -206.66 -293.98
network                             0.65  -1.17   -5.37   -5.75  -12.93  -17.22
non-power gas consumers            10.77  10.41   29.41   39.30   84.33  121.93
owners: H2 and DAC production      -0.00  -0.47   -0.45   -0.24   -0.25   -0.11
owners: SMR (new)                   9.05  11.04   11.49   18.13   11.72   11.42
owners: coal                       -0.45  -0.19   -2.58   -2.57   -6.70  -10.17
owners: existing nuclear           -4.19  -5.35  -16.05  -19.18  -43.12  -70.39
owne

In [12]:
def entries(out):
    ap = FREE(L.actor_pvs(out, True))
    ap = ap[(ap["funding"] == "T") & (ap["support_form"] == "lump") & (ap["horizon"] == "A")]
    return ap.groupby(["actor", "entry"])["PV_2.0%"].sum()

E_src = pd.DataFrame({t: entries(o) for t, o in PAIR_SRC.items()})[TARGETS]
E_fix = pd.DataFrame({t: entries(o) for t, o in PAIR_FIX.items()})[TARGETS]
rps_like = [i for i in E_src.index.union(E_fix.index) if ("RPS" in i[1]) or (i[1] == "revenue: rps") or ("ACP" in i[1])]
E = pd.concat({"source": E_src.reindex(rps_like), "fix": E_fix.reindex(rps_like)}, axis=1).fillna(0.0)
print("RPS-related entries, 2025 $B:")
print(E.round(2).to_string())

RPS-related entries, 2025 $B:
                                                                             source                                       fix                                   
                                                                                eia     aj   iaea    mck  cop28     eo    eia     aj   iaea    mck  cop28     eo
actor                             entry                                                                                                                         
electricity consumers             RPS payments                                13.97  19.23  38.07  34.04  32.31  24.33  19.20  27.79  47.68  46.82  50.94  34.47
foreign sellers: Canadian imports RPS trade wedge: seller's rule not binding   0.04   0.05   0.06   0.21   0.00   0.21  -0.04  -0.04  -0.04  -0.04  -0.04  -0.04
                                  RPS trade wedge: trade limits                0.00   0.00   0.00   0.00   0.00   0.00   0.00   0.00   0.00   0.00   0.00   0.00
    

In [13]:
def ref_views(out):
    v = FREE(L.views(out, True))
    v = v[(v["funding"] == "T") & (v["support_form"] == "lump") & (v["health_model"] == "AP2 ACS") & (v["mcpf"] == 1.0)
          & (v["phi"] == 0.0) & (v["scc_basis"] == "global EPA2023 2.0%")]
    return v.set_index("horizon")[["view1", "view2", "profits_domestic", "minus_d_resource_cost", "C3_residual"]]

VW = pd.concat({(w, t): ref_views(o) for w, P in (("source", PAIR_SRC), ("fix", PAIR_FIX)) for t, o in P.items()},
               names=["who", "target"])
VWd = VW.xs("fix", level="who") - VW.xs("source", level="who")
for h in ("A", "B"):
    x = pd.concat({"source": VW.xs("source", level="who").xs(h, level="horizon"),
                   "fix": VW.xs("fix", level="who").xs(h, level="horizon"),
                   "fix - source": VWd.xs(h, level="horizon")}, axis=1).loc[TARGETS]
    print(f"\nhorizon {h}, 2025 $B:")
    print(x[[c for c in x.columns if c[1] in ("view1", "view2", "profits_domestic")]].round(1).to_string())


horizon A, 2025 $B:
        source                             fix                         fix - source                       
         view1  view2 profits_domestic   view1  view2 profits_domestic        view1 view2 profits_domestic
target                                                                                                    
eia       41.7   -4.3            -46.2    49.9   13.7            -36.3          8.1  18.0              9.9
aj        66.0   -0.3            -66.5    77.7   17.6            -60.1         11.7  17.9              6.4
iaea     206.4   34.2           -172.7   222.5   52.7           -169.6         16.1  18.6              3.1
mck      313.0   98.1           -215.5   324.9  119.2           -205.5         11.9  21.1              9.9
cop28    780.4  313.9           -467.3   754.9  316.4           -438.4        -25.5   2.5             28.8
eo      1427.6  709.9           -718.6  1403.1  702.3           -701.6        -24.5  -7.6             17.0

horizon B, 2025

**The RPS account in the fix pairs.** Notebook 21's closure check for the RPS account: the RPS residual
(`P_R` − RPS revenue − ACP) and its trade-wedge term, summed 2026–2050 (2004 $B), source and fix.

In [14]:
rows = []
for w, P in (("source", PAIR_SRC), ("fix", PAIR_FIX)):
    for t, o in P.items():
        ck = o["checks"]
        for who in ("run", "base"):
            x = ck[(ck["who"] == who) & ck["t"].between(*Y)]
            rows.append(dict(who=w, target=t, run=who, RPS_residual=x["RPS_residual"].sum() / 1e9,
                             trade_wedge=x["RPS_trade_rent"].sum() / 1e9, RPS_gross=x["RPS_gross"].sum() / 1e9))
RPSC = pd.DataFrame(rows)
print(RPSC[RPSC["run"] == "run"].pivot_table(index="target", columns="who",
      values=["RPS_residual", "trade_wedge", "RPS_gross"]).loc[TARGETS].round(2).to_string())
b = RPSC[(RPSC["run"] == "base") & (RPSC["target"] == "eo")].set_index("who")[["RPS_residual", "trade_wedge", "RPS_gross"]]
print("\nbaseline fbR_none (source) and recfix_fbR_none (fix):")
print(b.round(2).to_string())

       RPS_gross        RPS_residual        trade_wedge       
who          fix source          fix source         fix source
target                                                        
eia        52.91  66.56        10.33  10.77       10.80  12.75
aj         49.52  64.21         9.80  10.33       10.24  12.32
iaea       43.34  58.39         9.23  10.27        9.57  12.04
mck        43.76  59.57         9.31  10.45        9.65  12.23
cop28      42.31  60.57         9.05  10.72        9.33  12.48
eo         47.86  63.27        10.37  11.44       10.62  13.38

baseline fbR_none (source) and recfix_fbR_none (fix):
        RPS_residual  trade_wedge  RPS_gross
who                                         
source         10.68        12.70      70.60
fix            10.40        10.93      58.46


## 5. Exports

| File | Contents |
|---|---|
| `run3_summary.csv` | every summary quantity of §1 for the 16 runs (source, fix, control) |
| `run3_reproduction.csv` | §2, control against source, with the fix's difference for scale |
| `run3_control_ledger.csv` | §2, ledger of each control against its source, actor sums |
| `run3_recs_by_origin.csv` | §3, the wedge and REC quantities by origin, by run, source and fix |
| `run3_recs_by_year.csv` | the wedge and REC quantities by origin per year, source, fix and control |
| `run3_other.csv` | §3, RPS payments, system cost, CO2, support |
| `run3_cap2050.csv` | §3, 2050 capacity by group, fix − source |
| `run3_fix_ledger.csv` | §3, ledger of each fix run against its source, actor sums |
| `run3_pair_actor.csv` | §4, actor sums of the mandate pairs, source and fix |
| `run3_pair_entries.csv` | §4, every entry of the mandate pairs, source and fix |
| `run3_pair_views.csv` | §4, reference views, both horizons |
| `run3_rps_account.csv` | §4, the RPS account in the source and fix pairs |

In [15]:
SUM.T.rename_axis(["who", "run"]).reset_index().to_csv(OUT / "run3_summary.csv", index=False)
REPRO.to_csv(OUT / "run3_reproduction.csv", index=False)
CL.rename_axis("actor").reset_index().to_csv(OUT / "run3_control_ledger.csv", index=False)
W.rename_axis(["who", "run"]).reset_index().to_csv(OUT / "run3_recs_by_origin.csv", index=False)
yr = pd.concat([rd(b, "derived_recs").query("case == @c").assign(run=k, who=w)
                for w, D_ in (("source", SRC), ("fix", FIX), ("control", CTRL)) for k, (b, c) in D_.items()])
yr = yr[yr["item"].str.startswith(("wedge|", "Q|")) & yr["t"].between(*Y)]
yr[["who", "run", "t", "item", "Value"]].to_csv(OUT / "run3_recs_by_year.csv", index=False)
OTHER.to_csv(OUT / "run3_other.csv", index=False)
CAP.rename_axis("group").reset_index().to_csv(OUT / "run3_cap2050.csv", index=False)
FL.rename_axis("actor").reset_index().to_csv(OUT / "run3_fix_ledger.csv", index=False)
pd.concat({"source": A_src, "fix": A_fix}, names=["who", "actor"]).reset_index().to_csv(OUT / "run3_pair_actor.csv", index=False)
pd.concat({"source": E_src, "fix": E_fix}, names=["who", "actor", "entry"]).reset_index().to_csv(OUT / "run3_pair_entries.csv", index=False)
VW.reset_index().to_csv(OUT / "run3_pair_views.csv", index=False)
RPSC.to_csv(OUT / "run3_rps_account.csv", index=False)
print("written:", sorted(p.name for p in OUT.glob("run3_*.csv")))

written: ['run3_cap2050.csv', 'run3_control_ledger.csv', 'run3_fix_ledger.csv', 'run3_other.csv', 'run3_pair_actor.csv', 'run3_pair_entries.csv', 'run3_pair_views.csv', 'run3_recs_by_origin.csv', 'run3_recs_by_year.csv', 'run3_reproduction.csv', 'run3_rps_account.csv', 'run3_summary.csv']


## 6. What reads these exports

Notebook 25 (`25_ag_review`) carries the RUN-3 results to the review page (the RPS residual page and Decisions,
item 3). Observations go to `docs/observations.md` by hand, with no interpretation (protocol §10).